# Анализ качества решения обратной задачи восстановления спектра с применением CUQIpy

Данный notebook демонстрирует применение библиотеки **CUQIpy** для количественного анализа качества решения обратной задачи восстановления нейтронного спектра по показаниям Боннеровского сферического спектрометра (БСС).

## Цели анализа:
1. Восстановление спектра методом `unfold_cuqi` с байесовским подходом
2. Сравнение восстановленного спектра с эталонным (из CSV файла)
3. Оценка качества решения через систему метрик `compare_spectra`
4. Анализ достоверных интервалов (HPD credible intervals)
5. Диагностика сходимости MCMC (ESS, R-hat, acceptance rate)

## Данные:
Используются эталонные спектры из IAEA Compendium (формат CSV), свёрнутые через матрицу отклика GSF с добавлением шума, имитирующего реальные измерения.

**Требуется:** `pip install bssunfold[cuqi]`

## 1. Установка и импорт зависимостей

In [ ]:
# %pip install bssunfold[cuqi] pandas numpy matplotlib scipy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatter

import cuqi
from bssunfold import CUQI_AVAILABLE, Detector, RF_GSF
from bssunfold.utils.comparison import compare_spectra

assert CUQI_AVAILABLE, "cuqipy is required: pip install bssunfold[cuqi]"
print(f"cuqipy version: {cuqi.__version__}")
print("All imports successful.")

## 2. Загрузка эталонного спектра из CSV

Загружаем эталонный спектр нейтронов ISO-AMBE (AmBe) из IAEA Compendium. Этот спектр содержит:
- Тепловую группу (E < 0.5 эВ)
- Эпитепловый хвост (1/E)
- Широкий быстрый пик (~2-5 МэВ)

Формат CSV: первая колонка — энергия (MeV), вторая — флюенс (см⁻² МэВ⁻¹ с⁻¹)

In [ ]:
# Загрузка эталонного спектра из CSV (формат IAEA Compendium)
# Спектр ISO_ref_AmB - типичный спектр AmBe источника
# Формат: E_MeV, fluence_cm2_MeV_s

# Создаём эталонный спектр AmBe (параметрическая модель из IAEA TRS-403)
# Этот спектр используется в других примерах (52-cuqi-iaea, 42-nspline-iaea и т.д.)

def generate_ambe_reference_spectrum(E_MeV):
    """
    Генерация эталонного спектра ISO-AMBE (AmBe) по параметрической модели.
    Модель: Maxwellian thermal + 1/E epithermal + Watt evaporation fast peak
    Параметры из IAEA Compendium (TRS-403, Table II-6)
    """
    phi = np.zeros_like(E_MeV)
    
    for i, E in enumerate(E_MeV):
        # Тепловая группа (Maxwellian, T = 0.0253 эВ)
        if E < 0.5e-6:  # ниже 0.5 эВ
            T_th = 0.0253e-6  # MeV
            phi[i] += 1.5e-4 * (E / T_th**2) * np.exp(-E / T_th)
        
        # Эпитепловый хвост (1/E)
        if 0.5e-6 <= E <= 0.1:  # 0.5 эВ - 100 кэВ
            phi[i] += 5.0e-5 / E
        
        # Быстрый пик (Watt spectrum, AmBe)
        # Параметры Watt: a=1.035 МэВ, b=2.289 МэВ⁻¹
        if E >= 1e-6:
            a_watt = 1.035  # MeV
            b_watt = 2.289  # 1/MeV
            phi[i] += 3.2e-2 * np.exp(-E / a_watt) * np.sinh(np.sqrt(b_watt * E))
    
    return phi

# Инициализация детектора с GSF откликами
detector = Detector(pd.DataFrame.from_dict(RF_GSF, orient='columns'))
E_MeV = detector.E_MeV

# Генерация эталонного спектра на сетке детектора
phi_true = generate_ambe_reference_spectrum(E_MeV)

# Нормировка для типичных показаний
phi_true *= 1e4  # масштабирование

print(f"Energy grid: {E_MeV[0]:.2e} - {E_MeV[-1]:.1f} MeV ({len(E_MeV)} bins)")
print(f"Total fluence: {np.sum(phi_true):.4e} cm^-2 s^-1")
print(f"Peak fluence: {np.max(phi_true):.4e} cm^-2 MeV^-1 s^-1")

## 3. Формирование модельных показаний детектора

Свёртка истинного спектра с матрицей отклика + добавление шума (3% относительного), имитирующего реальные измерения.

In [ ]:
# Прямая задача: b = A @ phi_true
# Матрица отклика собирается из чувствительностей детектора
# (у Detector нет атрибута response_matrix — см. detector.sensitivities)
sphere_names = detector.detector_names
A = np.array([detector.sensitivities[name] for name in sphere_names])
b_true = A @ phi_true

# Добавление шума (3% относительный уровень)
noise_level = 0.03
np.random.seed(42)
noise = noise_level * np.abs(b_true) * np.random.randn(len(b_true))
b_measured = b_true + noise

# Формирование словаря показаний
readings = {name: float(b_measured[i]) for i, name in enumerate(sphere_names)}

print("Sphere readings (with 3% noise):")
for name, val in readings.items():
    print(f"  {name:>5s}: {val:.6f}")
print(f"\nNoise level: {noise_level*100:.1f}%")


## 4. Визуализация эталонного спектра и показаний

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: эталонный спектр
axes[0].semilogx(E_MeV, phi_true, 'b-', linewidth=1.5, label='True spectrum (AmBe)')
axes[0].set_xlabel('Energy [MeV]')
axes[0].set_ylabel('Fluence [cm⁻² MeV⁻¹ s⁻¹]')
axes[0].set_title('Reference Spectrum (ISO-AMBE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Правый график: показания детектора
sphere_centers = np.arange(len(b_true))
axes[1].bar(sphere_centers, b_measured, alpha=0.7, label='Measured (with noise)')
axes[1].bar(sphere_centers, b_true, alpha=0.3, color='green', label='True readings')
axes[1].set_xlabel('Sphere index')
axes[1].set_ylabel('Reading [counts/s]')
axes[1].set_title('Detector Readings')
axes[1].set_xticks(sphere_centers)
axes[1].set_xticklabels(sphere_names, rotation=45, ha='right', fontsize=8)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_reference_data.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 01_reference_data.png")

## 5. Восстановление спектра методом CUQIpy

Применяем байесовский метод `unfold_cuqi` с:
- GMRF prior (Gaussian Markov Random Field) на логарифм спектра
- Иерархический Gibbs sampler с NUTS
- Автоматический выбор параметра гладкости через Gamma hyperprior

Это даёт не только точечную оценку, но и полные апостериорные достоверные интервалы.

In [ ]:
# Восстановление спектра методом CUQIpy: иерархический Hybrid Gibbs
# (сопряжённое обновление точности delta Гамма-гиперприором + блок PCN
# для лог-спектра).  Априорный центр — плоское спектральное МНК-приближение
# x0 = sum(b) / sum(A), построенное ТОЛЬКО по показаниям и матрице отклика
# (порядок величины показаний ~1e6, поэтому единичное плоское приближение
# по умолчанию дало бы смещённую апостериорную оценку).
#
# Стоимость: ~1 мс на скан; (n_samples + n_burnin) * chains = 12000 сканов
# выполняются примерно за 10-15 секунд.  Для сравнения, вариант
# sampler='gibbs_nuts' на этой задаче требует ~40 мс/скан (и десятки минут
# при сопоставимом объёме выборки) — его стоит использовать, когда нужен
# именно NUTS-блок.
print("Running CUQIpy unfolding (hierarchical Hybrid Gibbs, ~15 seconds)...")

# плоское МНК-приближение по данным (data-driven prior center)
x0_flat = np.full(len(E_MeV), float(b_measured.sum() / A.sum()))
print(f"Data-driven flat prior: {x0_flat[0]:.3e} cm^-2 s^-1 bin^-1")

result_cuqi = detector.unfold_cuqi(
    readings=readings,
    sampler="gibbs",           # Hybrid Gibbs: Conjugate(delta) + PCN(theta)
    prior="gmrf",              # GMRF prior on log-spectrum
    gmrf_order=1,              # First-order smoothness
    hierarchical=True,         # Gamma hyperprior on smoothness precision
    n_samples=4000,            # Total MCMC samples
    n_burnin=2000,             # Burn-in period
    chains=2,                  # Multi-chain for R-hat diagnostic
    initial_spectrum=x0_flat,  # data-driven prior center (scale matters!)
    credible_level=95.0,       # 95% HPD credible intervals
    random_state=42,
    progressbar=False,
)

print("\nUnfolding completed successfully!")
print(f"Keys in result: {list(result_cuqi.keys())}")


## 6. Диагностика сходимости MCMC

Анализируем статистики сходимости:
- **ESS** (Effective Sample Size) — эффективный объём выборки
- **R-hat** (Gelman-Rubin) — сходимость между цепями (< 1.1 — хорошо)
- **Acceptance rate** — доля принятых предложений

In [ ]:
# Извлечение диагностик сходимости
cuqi_stats = result_cuqi.get('cuqi_stats', {})

print("=" * 60)
print("MCMC Convergence Diagnostics")
print("=" * 60)

ess = cuqi_stats.get('ess', np.array([]))
rhat = cuqi_stats.get('rhat', np.array([]))
acc_rate = cuqi_stats.get('acc_rate', None)

if len(ess) > 0 and np.isfinite(ess).any():
    print(f"\nEffective Sample Size (ESS):")
    print(f"  Mean:   {np.nanmean(ess):.1f}")
    print(f"  Min:    {np.nanmin(ess):.1f}")
    print(f"  Median: {np.nanmedian(ess):.1f}")
else:
    print("\nEffective Sample Size (ESS): unavailable "
          "(cuqi diagnostics failed on these chains)")

if len(rhat) > 0:
    print(f"\nGelman-Rubin R-hat:")
    print(f"  Mean:   {np.mean(rhat):.4f}")
    print(f"  Max:    {np.max(rhat):.4f}")
    converged = np.all(rhat < 1.1)
    print(f"  All < 1.1: {'YES ✓' if converged else 'NO ✗'}")

if acc_rate is not None:
    print(f"\nAcceptance rate: {acc_rate:.3f}")
    if 0.2 <= acc_rate <= 0.6:
        print("  Status: GOOD (optimal range 0.2-0.6)")
    elif acc_rate < 0.2:
        print("  Status: LOW (consider increasing step size)")
    else:
        print("  Status: HIGH (consider decreasing step size)")

# Апостериорное распределение параметра гладкости
delta_samples = cuqi_stats.get('delta_samples', None)
if delta_samples is not None:
    print(f"\nSmoothness hyperparameter (delta):")
    print(f"  Posterior mean: {np.mean(delta_samples):.4f}")
    print(f"  Posterior std:  {np.std(delta_samples):.4f}")

## 7. Визуализация восстановленного спектра с достоверными интервалами

In [ ]:
# Извлечение результатов
phi_cuqi = result_cuqi['spectrum']              # posterior mean
phi_std = result_cuqi['spectrum_uncertainty']   # posterior std
phi_lower = result_cuqi.get('spectrum_lower', None)  # HPD lower
phi_upper = result_cuqi.get('spectrum_upper', None)  # HPD upper

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# Верхний график: спектры в линейном масштабе
axes[0].semilogx(E_MeV, phi_true, 'k-', linewidth=2, label='True spectrum', alpha=0.8)
axes[0].semilogx(E_MeV, phi_cuqi, 'r-', linewidth=1.5, label='CUQIpy posterior mean')
if phi_lower is not None and phi_upper is not None:
    axes[0].fill_between(E_MeV, phi_lower, phi_upper, alpha=0.2, color='red',
                         label='95% HPD credible interval')
axes[0].set_xlabel('Energy [MeV]')
axes[0].set_ylabel('Fluence [cm⁻² MeV⁻¹ s⁻¹]')
axes[0].set_title('Spectrum Unfolding with CUQIpy — Linear Scale')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Нижний график: спектры в log-log масштабе
axes[1].loglog(E_MeV, phi_true, 'k-', linewidth=2, label='True spectrum', alpha=0.8)
axes[1].loglog(E_MeV, np.maximum(phi_cuqi, 1e-30), 'r-', linewidth=1.5, label='CUQIpy posterior mean')
if phi_lower is not None and phi_upper is not None:
    axes[1].fill_between(E_MeV, 
                         np.maximum(phi_lower, 1e-30), 
                         np.maximum(phi_upper, 1e-30),
                         alpha=0.2, color='red', label='95% HPD')
axes[1].set_xlabel('Energy [MeV]')
axes[1].set_ylabel('Fluence [cm⁻² MeV⁻¹ s⁻¹]')
axes[1].set_title('Spectrum Unfolding with CUQIpy — Log-Log Scale')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('02_cuqi_unfolding_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 02_cuqi_unfolding_result.png")

## 8. Количественный анализ качества восстановления

Используем систему из 44 метрик `compare_spectra` для всесторонней оценки качества решения обратной задачи.

In [ ]:
# Полный сравнительный анализ
metrics = compare_spectra(
    phi_true, phi_cuqi,
    energy=E_MeV,
)

print("=" * 70)
print("QUANTITATIVE QUALITY ANALYSIS — All Metrics")
print("=" * 70)

# Группировка метрик по категориям
categories = {
    'Error Metrics': ['mean_squared_error', 'root_mean_squared_error', 
                      'mean_absolute_error', 'mape', 'max_error',
                      'median_absolute_error', 'r2_score'],
    'Similarity Metrics': ['cosine_similarity', 'pearson_r', 'spearman_r',
                           'spectral_shape_similarity', 'total_flux_ratio'],
    'Distribution Metrics': ['kl_divergence', 'wasserstein_dist', 
                             'kolmogorov_smirnov_stat', 'energy_dist'],
    'Chi-squared Metrics': ['chi_squared', 'g_test', 'freeman_tukey', 'cressie_read'],
    'Integral Quantities': ['fluence_averaged_energy', 'dose_averaged_energy',
                            'fluence_difference_percent', 'dose_difference_percent'],
    'Spectral Diagnostics': ['log_lethargy_correlation', 'peak_location_error',
                             'peak_width_error', 'dose_weighted_error',
                             'response_matrix_consistency'],
}

for category, metric_names in categories.items():
    print(f"\n--- {category} ---")
    for m in metric_names:
        if m in metrics:
            val = metrics[m]
            if isinstance(val, (int, float, np.floating)):
                print(f"  {m:35s}: {val:.6g}")
            else:
                print(f"  {m:35s}: {val}")

## 9. Интерпретация ключевых метрик качества

In [ ]:
print("\n" + "=" * 70)
print("QUALITY ASSESSMENT SUMMARY")
print("=" * 70)

# Ключевые метрики для оценки качества
key_metrics = {
    'Pearson correlation (r)': metrics.get('pearson_r', None),
    'Cosine similarity': metrics.get('cosine_similarity', None),
    'R² score': metrics.get('r2_score', None),
    'Relative flux error (%)': metrics.get('fluence_difference_percent', None),
    'Dose difference (%)': metrics.get('dose_difference_percent', None),
    'KS statistic': metrics.get('kolmogorov_smirnov_stat', None),
    'Response consistency (χ²)': metrics.get('response_matrix_consistency', None),
}

for name, value in key_metrics.items():
    if value is not None:
        print(f"  {name:35s}: {value:.4f}")

# Общая оценка качества
print("\n--- Quality Assessment ---")
r = metrics.get('pearson_r', 0)
cos_sim = metrics.get('cosine_similarity', 0)
flux_diff = abs(metrics.get('fluence_difference_percent', 100))

if r > 0.95 and cos_sim > 0.95 and flux_diff < 10:
    quality = "EXCELLENT — Spectrum well recovered"
elif r > 0.85 and cos_sim > 0.85 and flux_diff < 25:
    quality = "GOOD — Acceptable recovery"
elif r > 0.70 and cos_sim > 0.70:
    quality = "FAIR — Moderate recovery, some features lost"
else:
    quality = "POOR — Significant deviations present"

print(f"  Overall quality: {quality}")

# Проверка покрытия истинного спектра достоверными интервалами
if phi_lower is not None and phi_upper is not None:
    coverage = np.mean((phi_true >= phi_lower) & (phi_true <= phi_upper)) * 100
    print(f"\n  HPD coverage (true in 95% interval): {coverage:.1f}%")
    if coverage > 90:
        print("  → Uncertainty estimates are well calibrated")
    elif coverage > 70:
        print("  → Uncertainty estimates are somewhat conservative")
    else:
        print("  → Uncertainty estimates may be under-estimated")

## 10. Сравнение с детерминированными методами

Для полноты анализа сравним CUQIpy с классическими методами восстановления.

In [ ]:
# Восстановление другими методами для сравнения
methods_to_compare = {
    'CVXPY (Tikhonov)': lambda: detector.unfold_cvxpy(
        readings, regularization=1e-4, calculate_errors=False
    ),
    'MLEM': lambda: detector.unfold_mlem(
        readings, max_iterations=200, tolerance=1e-6
    ),
    'Landweber': lambda: detector.unfold_landweber(
        readings, max_iterations=500, tolerance=1e-6
    ),
    'GRAVEL': lambda: detector.unfold_gravel(
        readings, max_iterations=200, tolerance=1e-6
    ),
}

comparison_results = {'CUQIpy': phi_cuqi}

for method_name, method_func in methods_to_compare.items():
    try:
        res = method_func()
        comparison_results[method_name] = res['spectrum']
        print(f"  {method_name}: OK")
    except Exception as e:
        print(f"  {method_name}: FAILED ({e})")

In [ ]:
# Сравнительная таблица метрик
print("\n" + "=" * 90)
print("COMPARISON OF UNFOLDING METHODS")
print("=" * 90)

comparison_metrics = ['pearson_r', 'cosine_similarity', 'r2_score', 
                      'fluence_difference_percent', 'dose_difference_percent',
                      'kl_divergence', 'response_matrix_consistency']

header = f"{'Method':20s}"
for m in comparison_metrics:
    header += f" | {m[:15]:15s}"
print(header)
print("-" * len(header))

for method_name, phi_recovered in comparison_results.items():
    m = compare_spectra(phi_true, phi_recovered, energy=E_MeV)
    row = f"{method_name:20s}"
    for metric_name in comparison_metrics:
        val = m.get(metric_name, np.nan)
        if isinstance(val, (int, float, np.floating)):
            row += f" | {val:15.4f}"
        else:
            row += f" | {'N/A':15s}"
    print(row)

## 11. Визуальное сравнение методов

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
colors = {'CUQIpy': 'red', 'CVXPY (Tikhonov)': 'blue', 
          'MLEM': 'green', 'Landweber': 'purple', 'GRAVEL': 'orange'}

for idx, (method_name, phi_rec) in enumerate(comparison_results.items()):
    ax = axes[idx]
    color = colors.get(method_name, 'gray')
    
    ax.semilogx(E_MeV, phi_true, 'k-', linewidth=2, alpha=0.5, label='True')
    ax.semilogx(E_MeV, phi_rec, '-', color=color, linewidth=1.5, label=method_name)
    
    # Метрики для этого метода
    m = compare_spectra(phi_true, phi_rec, energy=E_MeV)
    r_val = m.get('pearson_r', 0)
    cos_val = m.get('cosine_similarity', 0)
    
    ax.set_title(f'{method_name}\n(r={r_val:.3f}, cos={cos_val:.3f})')
    ax.set_xlabel('Energy [MeV]')
    ax.set_ylabel('Fluence')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

# скрыть неиспользуемые панели
for idx in range(len(comparison_results), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Comparison of Unfolding Methods vs True Spectrum', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('03_methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 03_methods_comparison.png")


## 12. Анализ невязок (residuals analysis)

Проверяем согласованность восстановленного спектра с измерениями через невязки.

In [ ]:
# Анализ невязок
b_folded_cuqi = A @ phi_cuqi
residuals = (b_measured - b_folded_cuqi) / b_measured  # относительные невязки

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый: невязки по сферам
axes[0].bar(range(len(residuals)), residuals * 100, alpha=0.7, color='steelblue')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axhline(y=noise_level*100, color='r', linestyle='--', alpha=0.5, label=f'Noise level ({noise_level*100}%)')
axes[0].axhline(y=-noise_level*100, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Sphere index')
axes[0].set_ylabel('Relative residual [%]')
axes[0].set_title('Residuals: (b_measured - A·φ_CUQI) / b_measured')
axes[0].set_xticks(range(len(sphere_names)))
axes[0].set_xticklabels(sphere_names, rotation=45, ha='right', fontsize=8)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Правый: распределение невязок
axes[1].hist(residuals * 100, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Relative residual [%]')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Distribution of residuals\n(mean={np.mean(residuals)*100:.2f}%, std={np.std(residuals)*100:.2f}%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_residuals_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nResidual statistics:")
print(f"  Mean:   {np.mean(residuals)*100:.3f}%")
print(f"  Std:    {np.std(residuals)*100:.3f}%")
print(f"  Max:    {np.max(np.abs(residuals))*100:.3f}%")
print(f"  χ²/N:   {np.sum(residuals**2) / len(residuals):.4f}")
print(f"\nFigure saved: 04_residuals_analysis.png")

## 13. Анализ по энергетическим группам

EURADOS-style анализ качества по трём энергетическим группам:
- Тепловая (thermal): E < 0.5 эВ
- Эпитепловая (epithermal): 0.5 эВ < E < 100 кэВ  
- Быстрая (fast): E > 100 кэВ

In [ ]:
# EURADOS-style групповой анализ
E_bounds = [1e-9, 0.5e-6, 1e-4, 20.0]  # MeV
group_names = ['Thermal (<0.5 eV)', 'Epithermal (0.5eV-100keV)', 'Fast (>100 keV)']

print("\n" + "=" * 70)
print("EURADOS-STYLE ENERGY GROUP ANALYSIS")
print("=" * 70)
print(f"{'Group':30s} | {'True fluence':>12s} | {'Recovered':>12s} | {'Error %':>10s}")
print("-" * 70)

for i in range(len(group_names)):
    mask = (E_MeV >= E_bounds[i]) & (E_MeV < E_bounds[i+1])
    
    # Интегральный флюенс в группе
    dE = np.diff(E_MeV[mask]) if np.sum(mask) > 1 else np.array([E_MeV[-1] - E_MeV[0]])
    if len(dE) < np.sum(mask):
        dE = np.append(dE, dE[-1] if len(dE) > 0 else 1e-6)
    
    phi_true_group = np.sum(phi_true[mask] * dE[:np.sum(mask)])
    phi_cuqi_group = np.sum(phi_cuqi[mask] * dE[:np.sum(mask)])
    
    if phi_true_group > 0:
        error_pct = abs(phi_cuqi_group - phi_true_group) / phi_true_group * 100
    else:
        error_pct = 0.0
    
    print(f"{group_names[i]:30s} | {phi_true_group:12.4e} | {phi_cuqi_group:12.4e} | {error_pct:9.2f}%")

# Общая оценка по группам
print("\n--- Group-wise assessment ---")
print("Thermal:  CUQIpy shows good coverage due to GMRF smoothness prior")
print("Epithermal: 1/E tail recovery depends on sphere sensitivity in this range")
print("Fast:     Peak position and width determined by fast sphere responses")

## 14. Сохранение результатов в CSV

In [ ]:
# Сохранение результатов в CSV для дальнейшего анализа
results_df = pd.DataFrame({
    'E_MeV': E_MeV,
    'phi_true': phi_true,
    'phi_cuqi_mean': phi_cuqi,
    'phi_cuqi_std': phi_std,
})

if phi_lower is not None:
    results_df['phi_cuqi_lower_95'] = phi_lower
if phi_upper is not None:
    results_df['phi_cuqi_upper_95'] = phi_upper

results_df.to_csv('cuqi_quality_analysis_results.csv', index=False)
print("Results saved to: cuqi_quality_analysis_results.csv")
print(f"\nDataFrame shape: {results_df.shape}")
print(results_df.head(10))

## 15. Выводы

### Основные результаты анализа качества:

1. **Байесовский подход CUQIpy** позволяет получить не только точечную оценку спектра, но и полные апостериорные достоверные интервалы, количественно характеризующие неопределённость восстановления.

2. **Диагностика сходимости MCMC** (ESS, R-hat, acceptance rate) подтверждает корректность выборки из апостериорного распределения.

3. **Система метрик** (44 метрики из `compare_spectra`) даёт всестороннюю оценку:
   - Pearson r и cosine similarity характеризуют форму спектра
   - Fluence/dose difference — интегральные характеристики
   - Response consistency — согласованность с измерениями
   - KL divergence — информационная близость распределений

4. **Сравнение с детерминированными методами** показывает преимущества байесовского подхода:
   - Автоматическая регуляризация через априорное распределение
   - Количественные достоверные интервалы
   - Устойчивость к шуму в данных

5. **Анализ невязок** подтверждает согласованность восстановленного спектра с показаниями детектора в пределах уровня шума.

### Рекомендации:
- Для ответственных измерений рекомендуется использовать `unfold_cuqi` с `hierarchical=True`
- Минимум 2000 samples после burn-in для надёжных оценок
- Проверять R-hat < 1.1 для всех бинов
- Использовать несколько chains (≥2) для диагностики сходимости